In [ ]:
import os
import re
from tqdm import tqdm
from bs4 import BeautifulSoup
import time
import uuid
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import cloudscraper
from urllib.parse import quote_plus
from fake_useragent import UserAgent

# Initialize fake user agent generator
ua = UserAgent()

# Generate headers with a random User-Agent
headers = {
    "User-Agent": ua.random,
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}


scraper = cloudscraper.create_scraper()





def clean_title(title):
    """
    Remove parentheses and everything inside from the title.
    Example: 'The Diary of a Young Girl (Mass Market Paperback)' -> 'The Diary of a Young Girl'
    """
    return re.sub(r"\s*\(.*?\)", "", title).strip()

    
def downld_epub_fast(epub_link, scraper, download_dir="download_dir", chunk_size=65536, max_workers=4):
    """
    Fast EPUB downloader with multiple optimizations:
    - Larger chunk size (64KB default)
    - Parallel chunk downloading for large files
    - Reduced system calls
    - Optimized file I/O
    """
    try:
        os.makedirs(download_dir, exist_ok=True)
        
        # First, get file info with HEAD request (faster than GET for metadata)
        head_response = scraper.head(epub_link, timeout=30)
        head_response.raise_for_status()
        
        # Extract filename from Content-Disposition
        cd = head_response.headers.get("content-disposition", "")
        match = re.search(r'filename="?([^"]+)"?', cd)
        if match:
            raw_name = match.group(1)
            final_filename = os.path.basename(raw_name.strip('"'))
        else:
            print("❌ No valid filename in headers")
            return None
            
        save_path = os.path.join(download_dir, final_filename)
        
        # Skip if already exists
        if os.path.exists(save_path):
            print(f"⏭️  File already exists: {save_path}")
            return save_path
            
        total_size = int(head_response.headers.get("content-length", 0))
        
        # For small files or when parallel download isn't beneficial, use simple download
        if total_size < 10 * 1024 * 1024:  # Less than 10MB
            return _simple_fast_download(epub_link, scraper, save_path, final_filename, total_size, chunk_size)
        
            
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return None

def _simple_fast_download(epub_link, scraper, save_path, filename, total_size, chunk_size):
    """Optimized single-threaded download for smaller files"""
    with scraper.get(epub_link, stream=True, timeout=30) as response:
        response.raise_for_status()
        
        with open(save_path, "wb") as file, tqdm(
            desc=filename,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            # Write chunks in larger batches to reduce system calls
            buffer = bytearray()
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.extend(chunk)
                    # Write buffer when it gets large enough
                    if len(buffer) >= chunk_size * 4:  # Write every ~256KB
                        file.write(buffer)
                        bar.update(len(buffer))
                        buffer.clear()
            
            # Write remaining buffer
            if buffer:
                file.write(buffer)
                bar.update(len(buffer))
    
    print(f"✅ Download complete: {save_path}")
    return save_path


def fetch_and_download(payload, scraper=None, download_dir="download_dir", max_retries=3):
    """
    Given a payload {id, filename}, handle the whole process:
      - POST to Fetching_Resource.php
      - Extract redirect link
      - Validate headers
      - Download if valid
    Returns the saved file path or None.
    Retries HEAD request failures with exponential backoff (5s, 10s, 20s).
    """
    import re
    import time

    base_url = "https://oceanofpdf.com/Fetching_Resource.php"

    # Use provided scraper or create a new one
    scraper = scraper or cloudscraper.create_scraper()

    print(f"\n[+] Requesting resource for {payload['filename']}...")
    try:
        # Step 1: submit the form
        response = scraper.post(base_url, data=payload, timeout=20)
        response.raise_for_status()
    except Exception as e:
        print(f"❌ POST request failed: {e}")
        return None

    # Step 2: look for redirect link
    match = re.search(r'https://fs\d+\.oceanofpdf\.com/[^\s"\']+', response.text)
    if not match:
        print("[!] No redirect URL found. Response preview:")
        print(response.text[:500])
        return None

    redirect_url = match.group(0)

    # Step 3: HEAD request with retries
    attempt = 0
    head_resp = None
    while attempt < max_retries:
        try:
            head_resp = scraper.head(redirect_url, allow_redirects=True, timeout=15)
            head_resp.raise_for_status()
            break  # success, exit loop
        except Exception as e:
            attempt += 1
            if attempt < max_retries:
                wait_time = 5 * (2 ** (attempt - 1))  # 5s, 10s, 20s backoff
                print(f"❌ HEAD request failed (attempt {attempt}/{max_retries}): {e}")
                print(f"   Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"❌ HEAD request failed after {max_retries} attempts: {e}")
                return None

    # Step 4: validate content-disposition
    cd = head_resp.headers.get("content-disposition", "")
    if "attachment" in cd and "filename=" in cd:
        return downld_epub_fast(
            redirect_url, scraper,
            download_dir=download_dir,
            chunk_size=65536,
            max_workers=4
        )
    else:
        print("❌ No valid downloadable attachment in headers.")
        return None


def get_download_forms(book_url, scraper):
    """
    Fetch all download form details (id, filename) from a book page.
    Returns only EPUB forms if available, otherwise returns other formats.
    Skips forms if EPUB or PDF size > 10 MB.
    """
    try:
        response = scraper.get(book_url, headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {book_url}: {e}")
        return []
    
    soup = BeautifulSoup(response.text, "html.parser")

    # --- Extract file sizes ---
    def extract_size_in_mb(size_text):
        """
        Extract file size in MB from text like "24 MB" or "1.5 GB".
        Returns size in MB as float, or None if cannot parse.
        """
        import re
        
        # Clean the text and look for size patterns
        size_text = size_text.strip().upper()
        
        # Match patterns like "24 MB", "1.5 GB", "500 KB", etc.
        match = re.search(r'(\d+\.?\d*)\s*(MB|GB|KB)', size_text)
        
        if not match:
            return None
        
        size_value = float(match.group(1))
        unit = match.group(2)
        
        # Convert to MB
        if unit == "KB":
            return size_value / 1024
        elif unit == "MB":
            return size_value
        elif unit == "GB":
            return size_value * 1024
        
        return None

    entry_content = soup.find("div", class_="entry-content")
    if not entry_content:
        print(f"[!] Could not find entry-content div in {book_url}")
        #eturn []
        
    ul_tag = entry_content.find("ul")
    if not ul_tag:
        print(f"[!] Could not find ul tag in entry-content div in {book_url}")
        #eturn []
    #rint(ul_tag)
        # Extract file sizes
        
    pdf_size_mb = 0
    epub_size_mb = 0
    full_book_name = "Unknown"

    for li in ul_tag.find_all("li"):
        strong_text = li.find("strong")
        if strong_text:
            #rint(strong_text)
            text = strong_text.get_text().strip()
            if "Full Book Name" in text or "Full Book Name:" in text:
                full_book_name = li.get_text().replace(text, "").strip()
                #rint(full_book_name)
            if "PDF File Size" in text or "PDF File Size:" in text:
                # Extract size from the span or remaining text
                size_text = li.get_text().replace(text, "").strip()
                pdf_size_mb = extract_size_in_mb(size_text)
                #rint(size_text)
            elif "EPUB File Size:" in text or "EPUB File Size" in text:
                # Extract size from the span or remaining text
                size_text = li.get_text().replace(text, "").strip()
                epub_size_mb = extract_size_in_mb(size_text)

    # --- Skip if larger than 10 MB ---
    if pdf_size_mb > 10 or epub_size_mb > 10:
        print(f"[!] Skipping {full_book_name} (PDF: {pdf_size_mb} MB, EPUB: {epub_size_mb} MB) too large")
        return []

    # --- Process forms ---
    forms = soup.find_all("form", action="https://oceanofpdf.com/Fetching_Resource.php")
    epub_forms, other_forms = [], []

    for form in forms:
        id_input = form.find("input", {"name": "id"})
        filename_input = form.find("input", {"name": "filename"})
        
        if id_input and filename_input:
            file_ext = filename_input["value"].split(".")[-1].lower()
            form_data = {
                "id": id_input["value"],
                "filename": filename_input["value"]
            }
            if file_ext == "epub":
                epub_forms.append(form_data)
            else:
                other_forms.append(form_data)

    time.sleep(3)  # throttle requests
    return epub_forms if epub_forms else other_forms



def get_last_page(url):
    """Find the last page number from pagination."""
    try:
        print(f"getting last page for {url}")
        response = scraper.get(url,headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {url}: {e}")
        return 1  # fallback: only page 1

    soup = BeautifulSoup(response.text, "html.parser")
    pagination_div = soup.find("div", class_="archive-pagination pagination")

    if not pagination_div:
        return 1

    page_numbers = []
    for a_tag in pagination_div.find_all("a", href=True):
        # remove <span> tags
        for span in a_tag.find_all("span"):
            span.decompose()

        text = a_tag.get_text(strip=True)
        if text.isdigit():
            page_numbers.append(int(text))

    time.sleep(3)

    return max(page_numbers) if page_numbers else 1




def Ocean_of_pdf_search_books(search_query, start_page=None, stop_page=None, max_pages=None, first_only=False, first_n_books=None):
    base_url = "https://oceanofpdf.com/page/1/?s="
    
    def log_failed_book(entry):
        """Immediately append a failed book entry to log file."""
        os.makedirs("logs", exist_ok=True)
        fail_log = os.path.join("logs", "failed_books_11.txt")
        mode = "a" if os.path.exists(fail_log) else "w"
        with open(fail_log, mode, encoding="utf-8") as f:
            if mode == "w":  # first time create
                f.write(f"--- New session: {time.strftime('%Y-%m-%d %H:%M:%S')} ---\n")
            f.write(str(entry) + "\n")
        print(f"📄 Logged failed book immediately: {entry}")

    failed_books = []  # <--- Track failed book URLs here

    if 'by' in search_query:
        parts = search_query.split(" by ")
        title = parts[0]
        author = parts[1] if len(parts) > 1 else None 
    else:
        author = search_query

    full_url = f"{base_url}{quote_plus(search_query)}"
    last_page = get_last_page(full_url)
    print(f"Detected last page: {last_page}")

    start_page = start_page or 1
    stop_page = stop_page or last_page
    if max_pages:
        stop_page = min(start_page + max_pages - 1, stop_page)

    print(f"Fetching from page {start_page} to {stop_page}")
    books_downloaded = 0  

    for page in range(start_page, stop_page + 1):
        page_url = f"https://oceanofpdf.com/page/{page}/?s={quote_plus(search_query)}"
        print(f"[+] Fetching page {page}: {page_url}")

        try:
            response = scraper.get(page_url, headers=headers, timeout=15)
            response.raise_for_status()
        except Exception as e:
            print(f"[!] Failed to fetch {page_url}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        articles = soup.find_all("article")

        if first_only:
            articles = articles[:1]
        elif first_n_books is not None:
            remaining = first_n_books - books_downloaded
            if remaining <= 0:
                break
            articles = articles[:remaining]

        for article in articles:
            header = article.find("header", class_="entry-header")
            if not header:
                continue
            a_tag = header.find("a", class_="entry-title-link", href=True)
            if not a_tag:
                continue
        
            # Extract Language name from postmetainfo div
            postmetainfo = article.find("div", class_="postmetainfo")
            if postmetainfo:
                # ✅ Language filtering
                language_strong = postmetainfo.find("strong", string="Language: ")
                if language_strong:
                    language_text = language_strong.next_sibling
                    if language_text and language_text.strip().lower() != "english":
                        print(f"❌ Skipping non-English book")
                        continue  

                """ 
                # ✅ Author filtering
                author_strong = postmetainfo.find("strong", string="Author: ")
                if author_strong:
                    author_text = author_strong.next_sibling
                    if author_text:
                        book_author = author_text.strip().lower()
                        if author and author.strip().lower() not in book_author.strip().lower():
                            print(f"❌ Skipping book by '{book_author}' (looking for '{author}')")
                            continue
                
                """                 
                book_url = a_tag["href"]
                payload_list = get_download_forms(book_url, scraper)
                if not payload_list:
                    print("❌ No forms found on page or skipped due to size limit")
                    #ailed_books.append({"url": book_url, "reason": "No forms found"})
                    continue
    
                if isinstance(payload_list, dict):
                    result = fetch_and_download(payload_list, scraper, download_dir=f"download_{author}")
                    if result:
                        books_downloaded += 1
                        print(f"📚 Books downloaded so far: {books_downloaded}")
                    else:
                        failed_books.append({"url": book_url, "filename": payload_list.get("filename", "Unknown")})
    
                elif isinstance(payload_list, list):
                    success = False
                    for payload in payload_list:
                        result = fetch_and_download(payload, scraper, download_dir=f"download_{author}")
                        if result:
                            books_downloaded += 1
                            success = True
                            print(f"📚 Books downloaded so far: {books_downloaded}")
                            break
                    if not success:
                        failed_books.append({"url": book_url, "filenames": [p["filename"] for p in payload_list]})
    
                if first_only or (first_n_books is not None and books_downloaded >= first_n_books):
                    break

        time.sleep(5)

    print(f"\n✅ Total books downloaded: {books_downloaded}")
    print(f"❌ Failed books: {len(failed_books)}")

    # Save failed attempts for retry
    if failed_books:
        os.makedirs("logs", exist_ok=True)
        fail_log = os.path.join("logs", "failed_books_11.txt")
        
        # Append if file exists, create if it doesn't
        mode = "a" if os.path.exists(fail_log) else "w"
        action = "appended to" if mode == "a" else "saved to"
        
        with open(fail_log, mode, encoding="utf-8") as f:
            # Add timestamp header when appending
            if mode == "a":
                f.write(f"\n--- New session: {time.strftime('%Y-%m-%d %H:%M:%S')} ---\n")
            
            for entry in failed_books:
                f.write(str(entry) + "\n")
        
        print(f"📄 Failed book URLs {action} {fail_log}")
        
def download_failed_books(failed_log_file):
    """
    Read failed book entries from log file and attempt to re-download them.
    Each entry is expected to be a dictionary with at least a 'url' key.
    """
    if not os.path.exists(failed_log_file):
        print(f"[!] Failed log file does not exist: {failed_log_file}")
        return

    with open(failed_log_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    failed_entries = []
    for line in lines:
        line = line.strip()
        if line.startswith("{") and line.endswith("}"):
            try:
                entry = eval(line)  # ⚠️ Caution: using eval, ensure log integrity
                if isinstance(entry, dict) and "url" in entry:
                    failed_entries.append(entry)
            except Exception as e:
                print(f"[!] Failed to parse line: {line} | Error: {e}")

    if not failed_entries:
        print("[*] No valid failed entries found in log.")
        return

    total = len(failed_entries)
    print(f"[*] Found {total} failed entries to retry.")

    scraper = cloudscraper.create_scraper()
    re_failed = []

    for idx, entry in enumerate(failed_entries, start=1):
        book_url = entry["url"]
        print(f"\n[+] Retrying book: {book_url}")

        payload_list = get_download_forms(book_url, scraper)
        if not payload_list:
            print("❌ ebook size too large or No forms found on page")
            
            re_failed.append({"url": book_url, "reason": "No forms found"})
        else:
            if isinstance(payload_list, dict):
                result = fetch_and_download(payload_list, scraper, download_dir="retry_downloads")
                if not result:
                    re_failed.append({"url": book_url, "filename": payload_list.get("filename", "Unknown")})

            elif isinstance(payload_list, list):
                success = False
                for payload in payload_list:
                    result = fetch_and_download(payload, scraper, download_dir="retry_downloads")
                    if result:
                        success = True
                        break
                if not success:
                    re_failed.append({"url": book_url, "filenames": [p["filename"] for p in payload_list]})

        # ✅ Progress counter
        print(f"[*] Processed {idx} of {total} failed entries")

        time.sleep(5)

    print(f"\n✅ Retry complete. Successfully downloaded {total - len(re_failed)} out of {total} books.")


In [ ]:
logs_dir = r"C:\cracks\My_App\epub-library-manager\logs"
for log_file in os.listdir(logs_dir):
    if log_file.startswith("failed_books_") and log_file.endswith(".txt"):
        print(f"\n=== Processing log file: {log_file} ===")
        download_failed_books(os.path.join(logs_dir, log_file))

        # After finishing each book log file, delete it
        os.remove(os.path.join(logs_dir, log_file))

In [ ]:
folder='Boxset'
print(f"Searching for: {folder} books")
Ocean_of_pdf_search_books(search_query=folder,first_n_books=5)






In [ ]:
import os

path = r"C:\cracks\My_App\epub-library-manager\upload"

result = []

for name in os.listdir(path):
    folder_path = os.path.join(path, name)
    if os.path.isdir(folder_path):
        # Count only .pdf and .epub files inside the folder (not subfolders)
        count = sum(
            1 for f in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, f)) and f.lower().endswith((".pdf", ".epub"))
        )
        
        if count < 10:
            result.append(name.replace("_", " "))

for folder in result:
    print(f"Searching for: {folder} books")
    #Ocean_of_pdf_search_books(search_query=folder)

In [ ]:
import re

def extract_urls_from_logs(log_file):
    """
    Reads a log file and returns a list of URLs found in lines containing 'url'.
    """
    urls = []
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            match = re.search(r"'url':\s*'([^']+)'", line)
            if match:
                urls.append(match.group(1))
    return urls
failed_log_file = r"C:\cracks\My_App\epub-library-manager\logs\failed_books_11.txt"
failed_urls = extract_urls_from_logs(failed_log_file)


In [ ]:
failed_urls